## Description

Wav2vec2 pretraining is not phoneme-supervised.

It learns:
- masked acoustic modeling
- contextual acoustic representations

Phoneme structure emerges implicitly.

But explicit phoneme fine-tuning can refine that structure.

- wav2vec2 is an acoustic model, however, for "utility" in downstream application, it is not enough to only predict a continue string of phoneme, this string of phonetic sound need to be split into words, however, acoustic clues does not always provide that information (heavily dense speech, sometime does not let acoustic gaps between words). This means that the separation of word (an sylables, etc) is at a language level, not in an acoustic level; **meaning** language model to transfrom string of phoneme to words needs to be explore.

- there are two stage of wav2vec2 training
    1. self-supervised pre-trained (no CTC head)
    2. Supervise fine-tuning ASR.
Hugging Face has multiple option of fine-tuning, the one that I am using is this one

- Freeze the CNN, train transformer + head (most common): moderate dataset
    - `model.freeze_feature_encoder()` (only free CNN part of the wav2vec2)


- Train only the CTC head (feeze all backbone): for small dataset to avoid overffiting
    ```python
    for p in model.wav2vec2.parameters():
        p.requires_grad = False
    # head stays trainable (lm_head / classifier)
    ```

- Freeze most layers, unfreeze only top Transformer layers (“gradual unfreezing”)

```python
# freeze everything first
for p in model.wav2vec2.parameters():
    p.requires_grad = False

# unfreeze last N transformer layers
N = 4
for layer in model.wav2vec2.encoder.layers[-N:]:
    for p in layer.parameters():
        p.requires_grad = True

# keep head trainable
for p in model.lm_head.parameters():
    p.requires_grad = True

```

- Freeze most layers, unfreeze only top Transformer layers (“gradual unfreezing”)

```python
for p in model.parameters():
    p.requires_grad = True
````

-------
3) “Partially modify backbone” without fully unfreezing it
Option 1: Different learning rates (discriminative LR)

You can train all layers but make backbone updates tiny:

Head LR = 1e-4

Transformer LR = 1e-5

CNN LR = 1e-6

This is done with parameter groups in your optimizer.

Option 2: Adapters / LoRA (PEFT)

Instead of changing the original backbone weights, you add small trainable modules (adapters or LoRA) and train those.
This is often great for small datasets and avoids catastrophic forgetting.

(If you want, tell me your exact HF model class and I’ll give you a working PEFT/LoRA snippet.)
    
  

**XLS-R** is a multilingual self-supervised speech representation model developed by Meta AI. It extends the wav2vec 2.0 framework to cover over 100 languages, enabling robust speech recognition and understanding across diverse linguistic contexts. XLS-R is notable for its scale, versatility, and impact on low-resource language processing

**XLSR-53** is a large multilingual self-supervised speech representation model in the wav2vec2-large-xlsr-53 family. It extends wav2vec 2.0 to 53 languages, enabling powerful transfer to automatic speech recognition (ASR) and related tasks, especially in low-resource settings. It was introduced by researchers at Meta AI.

Strategy 1 for cross-language pre-trained model
1. Pretrain wav2vec2 self-supervised on large multilingual audio
2. Fine-tune with grapheme (character) CTC on each language

Example:

- XLS-R
- XLSR-53

These models are pretrained on 50+ languages.

Strattegy 2
1. Take pretrained wav2vec2 backbone.
2. Fine-tune on a language that has phoneme transcripts
3. Fine-tune again on low-resource language.
    
    Two options here:
        A) Keep phoneme output
        B) Switch to grapheme output

## take away

- I need a XLSR (base and CTC) fine-tune in more that 50 language

If your goal is:

✅ Clinical phonetic boundary precision

→ HMM forced alignment (MFA) is still very strong.

✅ Robust cross-domain / noisy speech

→ wav2vec2 alignment is stronger.

✅ Multilingual / low-resource

→ wav2vec2 multilingual (e.g., XLS-R) is better.

Wav2vec2 is a good feature extractor, but for audio-only “phoneme-like” units, many people prefer HuBERT-style representations because they were designed to learn more discrete, phone-like structure (still self-supervised). Either can work.

If your goal is real phonemes (IPA/ARPAbet) with timestamps from audio only, then you need a phone recognizer, which is basically “ASR but with phone targets”. Without any supervision, you’ll get pseudo-phones, not guaranteed phonemes.

## Import libraries

## Proto LM 1 

In [1]:
phoneme_sequence = ["sh", "iy", "hv", "ae", "d"]

# toy pronunciation dictionary
pron_dict = {
    "she": ["sh", "iy"],
    "had": ["hv", "ae", "d"]
}

In [2]:


def segment_phonemes(phonemes, pron_dict):
    result = []
    i = 0

    while i < len(phonemes):
        matched = False
        for word, pron in pron_dict.items():
            L = len(pron)
            if phonemes[i:i+L] == pron:
                result.append(word)
                i += L
                matched = True
                break
        if not matched:
            result.append(f"[UNK:{phonemes[i]}]")
            i += 1

    return result




In [3]:
segment_phonemes(phoneme_sequence, pron_dict)

['she', 'had']

In [4]:
phonemes = ["sh", "iy", "hv", "ae", "d"]

pron_dict = {
    "she": ["sh", "iy"],
    "had": ["hv", "ae", "d"],
    "heed": ["hh", "iy", "d"]
}

from functools import lru_cache

@lru_cache(None)
def segment(i):
    if i == len(phonemes):
        return []

    for word, pron in pron_dict.items():
        L = len(pron)
        if phonemes[i:i+L] == pron:
            rest = segment(i+L)
            if rest is not None:
                return [word] + rest

    return None

print(segment(0))

['she', 'had']
